In [ ]:
# 클래스 가중치 사용
# 리샘플링 기법
# 적절한 평가지표

In [ ]:
# 불균형 데이터 생성 (1:9) 악성0 : 양성1
from sklearn.datasets import load_breast_cancer
import numpy as np
np.random.seed(42)  # 같은 코드를 실행하면 항상 같은 "무작위 선택"결과가 나옴. 
# 이걸 고정해야지 최종 출력값이 항상 동일/ 변경이 안됨. 

data = load_breast_cancer()
X, y = data.data, data.target

#악성을 소수 클래스로 생성
print(f'악성 양성의 오리지널 비율 : {np.unique(y,return_counts=True)}') #return_counts=True → 각 값이 몇 번 나왔는지도 같이 알려줌
m_index = np.where(y==0)[0]  #index 반환 악성  #np.where()은 항상 튜플(array([0, 2]),)로 반환되므로, [0] 기재해야지 array([0, 2, 5]) 이렇게 나옴.. 실제 인덱스값
b_index = np.where(y==1)[0]

# 악성은 일부만, 양성은 더 많이 사용
# 약성의 30%만 사용, 양성은 전체 1.5 : 8.5
size_30 = int(len(m_index)*0.2) #int(len(m_index)*0.3) 1:9로 맞추기 위해 0.3에서 0.2로 변경
selected_m_index = np.random.choice(m_index, size=size_30, replace=False) #무작위 샘플 뽑음. replace=False 중복없이
selected_b_index = b_index  #변수명 통일위해서
# np.random.sample
len(selected_b_index) / (len(selected_m_index) + len(selected_b_index))  #출력 0.85

concatenate_selected_index = np.concatenate([selected_m_index, selected_b_index]) #양성과 악성 인덱스 합치기
np.random.shuffle(concatenate_selected_index) #shuffle 반환하는건 없고 섞어만줌.. 

X_imb = X[concatenate_selected_index]
y_imb = y[concatenate_selected_index]

# 클래스 분포 확인
unique, counts = np.unique(y_imb, return_counts=True)
print('클래스 분포')
for label, count in zip(unique, counts):
    percentage = count / len(y_imb)*100
    print(f'클래스 {label} : {count}개 ({percentage: .1f}%)')



# 불균형을 무시하고 모델의 성능을 평가
# 스케일링 정규화 StandardScaler
# LogisticRegression
# Pipe
# 평가는 class report


from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


X_train, X_test, y_train, y_test = train_test_split(X_imb, y_imb, stratify=y_imb, test_size=0.2, random_state=42)


print('1. 기본모델 (불균형 무시)')
pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(random_state=42, max_iter=1000))
])
pipe.fit(X_train,y_train)
print(classification_report(y_test, pipe.predict(X_test), target_names=['악성(0)', '양성(1)']))



악성 양성의 오리지널 비율 : (array([0, 1]), array([212, 357]))
클래스 분포
클래스 0 : 42개 ( 10.5%)
클래스 1 : 357개 ( 89.5%)
1. 기본모델 (불균형 무시)
              precision    recall  f1-score   support

       악성(0)       1.00      0.88      0.93         8
       양성(1)       0.99      1.00      0.99        72

    accuracy                           0.99        80
   macro avg       0.99      0.94      0.96        80
weighted avg       0.99      0.99      0.99        80



In [119]:
print('2. 불균형 해결 : 클래스 가중치 사용')

from copy import deepcopy
pipe_weight = deepcopy(pipe)  #deep copy 해도 동일하므로... 여기서는 deepcopy 필요없음.
# pipe_weight = Pipeline([
#             ('scaler', StandardScaler()),
#             ('clf', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
# ])

# 기존 파이프라인(상기#)에 기재한 clf 이름을 가진 객체의 파라미터만 조정가능함.
pipe_weight = pipe_weight.set_params(clf__class_weight='balanced')
pipe_weight.fit(X_train, y_train)
print(classification_report(y_test, pipe_weight.predict(X_test), target_names=['악성(0)', '양성(1)']))


2. 불균형 해결 : 클래스 가중치 사용
              precision    recall  f1-score   support

       악성(0)       1.00      0.88      0.93         8
       양성(1)       0.99      1.00      0.99        72

    accuracy                           0.99        80
   macro avg       0.99      0.94      0.96        80
weighted avg       0.99      0.99      0.99        80



In [121]:
print('3. 가중치 계산')

n_samples = len(y_train)
n_classes = 2
class_counts = np.bincount(y_train)
class_counts
for i in range(n_classes):
    weight = n_samples/ (n_classes*class_counts[i])
    print(f'클래스 : {i} 에 대한 가중치 {weight:.3f}')

3. 가중치 계산
클래스 : 0 에 대한 가중치 4.691
클래스 : 1 에 대한 가중치 0.560


In [ ]:
print('4. 불균형 해결 : RandomForest 모델 이용')

from sklearn.ensemble import RandomForestClassifier
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])
pipe_rf.fit(X_train,y_train)
print(f'불균형 모드 : {classification_report(y_test, pipe_rf.predict(X_test))}')

#모든 샘플을 동일한 중요도로 학습합니다.
# 그러면 데이터가 2:8로 불균형할 때,
# 모델은 대부분의 샘플(양성, 1)을 맞추는 쪽으로 최적화돼요.
# 즉, “양성만 잘 맞추고 악성은 거의 무시”. ==> 그래서 f1-score값이 아래 balanced 보다 더 높게 나옴


pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, max_depth=5))
])                                                              #max_depth 하든 안하든 결과 동일
pipe_rf.fit(X_train,y_train)
print(f'균형모드 : {classification_report(y_test, pipe_rf.predict(X_test))}')

# => 균형모드(class_weight='balanced') f1-score값이  더 안좋은 이유는 
# class_weight='balanced'를 썼더니 f1-score가 더 낮아진 이유는,
# 모델이 다수 클래스(양성) 성능을 일부러 떨어뜨려서
# 소수 클래스(악성) 도 더 잘 맞추려 한 결과예요.
# 즉, 정확도는 낮아지지만 공정한 모델이 된 것이에요.


4. 불균형 해결 : RandomForest 모델 이용
불균형 모드 :               precision    recall  f1-score   support

           0       1.00      0.75      0.86         8
           1       0.97      1.00      0.99        72

    accuracy                           0.97        80
   macro avg       0.99      0.88      0.92        80
weighted avg       0.98      0.97      0.97        80

균형모드 :               precision    recall  f1-score   support

           0       1.00      0.62      0.77         8
           1       0.96      1.00      0.98        72

    accuracy                           0.96        80
   macro avg       0.98      0.81      0.87        80
weighted avg       0.96      0.96      0.96        80



In [ ]:
# over / under sampling  직접 실행 SMOTE SMOTEEN 